In [6]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.parser import build_miner
from src.explain import build_baseline, explain_session

X_counts = np.load('../data/features/X_counts_full.npy')
y = np.load('../data/features/y_full.npy')
sessions = pd.read_csv('../data/features/sessions.csv')

# Load the templates learned during the full-dataset run
miner = build_miner('../drain3.ini', '../models/drain_state_full.bin')
template_map = {c.cluster_id: c.get_template() for c in miner.drain.clusters}

event_names = sorted(template_map.keys())
baseline = build_baseline(X_counts, y)

print("Templates:", len(event_names), "| Matrix columns:", X_counts.shape[1])
assert len(event_names) == X_counts.shape[1], "Mismatch between templates and matrix"

Templates: 45 | Matrix columns: 45


In [7]:
pd.DataFrame({'EventId': event_names,
              'EventTemplate': [template_map[e] for e in event_names]}
             ).to_csv('../data/parsed/templates_full.csv', index=False)
print("Saved")

Saved


In [9]:
anomaly_rows = np.where(y == 1)[0][:3]

for row in anomaly_rows:
    result = explain_session(X_counts[row], baseline, event_names)
    print(f"\n=== {sessions.iloc[row]['BlockId']} ===")

    print("Occurred more than expected:")
    for item in result['excess']:
        print(f"  {template_map[item['event']][:60]}")
        print(f"     {item['observed']:.0f}x  (expected {item['expected']})")

    print("Missing:")
    for item in result['missing']:
        print(f"  {template_map[item['event']][:60]}")
        print(f"     {item['observed']:.0f}x  (expected {item['expected']})")


=== blk_-104087085791207724 ===
Occurred more than expected:
  writeBlock <BLK> received exception java.io.IOException: Cou
     1x  (expected 0.0)
Missing:
  BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is 
     0x  (expected 3.01)
  Received block <BLK> of size <NUM> from /<IP>
     0x  (expected 3.0)
  PacketResponder <NUM> for block <BLK> <*>
     0x  (expected 3.0)
  Deleting block <BLK> file <PATH><BLK>
     0x  (expected 2.45)
  BLOCK* NameSystem.delete: <BLK> is added to invalidSet of <I
     0x  (expected 2.44)

=== blk_-1065365386304003658 ===
Occurred more than expected:
  <IP>:Got exception while serving <BLK> to /<IP>:
     2x  (expected 0.62)
  BLOCK* NameSystem.addStoredBlock: addStoredBlock request rec
     1x  (expected 0.0)
  BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is 
     4x  (expected 3.01)
  Verification succeeded for <BLK>
     1x  (expected 0.21)
  BLOCK* NameSystem.delete: <BLK> is added to invalidSet of <I
     3x  (expected 2.44)

In [11]:
from src.diagnose import diagnose

for row in anomaly_rows:
    explanation = explain_session(X_counts[row], baseline, event_names)
    result = diagnose(explanation, template_map)

    print(f"\n=== {sessions.iloc[row]['BlockId']} ===")
    print("Primary cause:", result['primary_cause'])
    print("All causes:", result['all_causes'])
    for line in result['evidence']:
        print("  •", line)


=== blk_-104087085791207724 ===
Primary cause: INCOMPLETE_EXECUTION
All causes: ['INCOMPLETE_EXECUTION', 'APPLICATION_ERROR']
  • writeBlock <BLK> received exception java.io.IOException: Could not rea occurred 1x (expected 0.0)
  • BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to < missing (expected 3.01)
  • Received block <BLK> of size <NUM> from /<IP> missing (expected 3.0)
  • PacketResponder <NUM> for block <BLK> <*> missing (expected 3.0)

=== blk_-1065365386304003658 ===
Primary cause: APPLICATION_ERROR
All causes: ['APPLICATION_ERROR']
  • <IP>:Got exception while serving <BLK> to /<IP>: occurred 2x (expected 0.62)
  • BLOCK* NameSystem.delete: <BLK> is added to invalidSet of <IP> occurred 3x (expected 2.44)

=== blk_-1347949337410740470 ===
Primary cause: APPLICATION_ERROR
All causes: ['APPLICATION_ERROR']
  • <IP>:Got exception while serving <BLK> to /<IP>: occurred 4x (expected 0.62)


In [12]:
from src.narrate import narrate
from src.diagnose import diagnose

for row in anomaly_rows[:3]:
    explanation = explain_session(X_counts[row], baseline, event_names)
    diagnosis = diagnose(explanation, template_map)
    print(narrate(explanation, diagnosis, template_map,
                  sessions.iloc[row]['BlockId']))
    print('\n' + '-' * 70 + '\n')

Session: blk_-104087085791207724
Severity: HIGH

What happened: The operation started but never finished

Why we think so:
  - "WriteBlock received exception Could not read from stream" happened once, but normally never happens at all.
  - "BlockMap updated is added to size" never happened, although it normally happens 3 times.
  - "Received block of size from" never happened, although it normally happens 3 times.
  - "PacketResponder for block" never happened, although it normally happens 3 times.

What this usually means:
  Steps that normally happen every single time did not happen at all. The process was interrupted partway through.

What to check:
  Check whether the process was killed, the machine restarted, or a dependency became unavailable mid-operation.

----------------------------------------------------------------------

Session: blk_-1065365386304003658
Severity: MEDIUM

What happened: The software reported an error

Why we think so:
  - "Got exception while serving to" 